In [1]:
"""
Funda scraper:
- Search pagina's via ?page=
- Detail links verzamelen
- Foto's via /media/foto/1..N (direct downloaden)
- Features scrapen en opslaan als JSON per listing_id
- Skip: als features.json al bestaat, sla listing over
- first_only: als True -> download alleen foto/1 en ga door naar volgende listing

Install:
  pip install -U selenium pandas requests
"""

import os
import re
import time
import json
import random
from pathlib import Path
from urllib.parse import urlparse, urlunparse, parse_qs, urlencode, urljoin

import pandas as pd
import requests
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import TimeoutException


#SEARCH_URL = "https://www.funda.nl/en/zoeken/koop?selected_area=%5B%22geldrop%22%5D"
SEARCH_URL = "https://www.funda.nl/en/zoeken/koop?object_type=[%22house%22]&availability=[%22available%22]"

# -------------------------
# Utils
# -------------------------
def _sleep(a: float = 0.8, b: float = 1.6) -> None:
    time.sleep(random.uniform(a, b))


def _safe_mkdir(p: Path) -> None:
    p.mkdir(parents=True, exist_ok=True)


def _try_accept_cookies(driver) -> None:
    candidates = [
        (By.XPATH, "//button[contains(translate(., 'ACCEPTEER', 'accepteer'), 'accepteer')]"),
        (By.XPATH, "//button[contains(translate(., 'AKKOORD', 'akkoord'), 'akkoord')]"),
        (By.XPATH, "//button[contains(., 'Accept all')]"),
        (By.XPATH, "//button[contains(., 'I agree')]"),
    ]
    for by, sel in candidates:
        try:
            btns = driver.find_elements(by, sel)
            if btns:
                btns[0].click()
                time.sleep(0.8)
                return
        except Exception:
            pass


def _make_driver(headless: bool) -> webdriver.Chrome:
    opts = Options()
    if headless:
        opts.add_argument("--headless=new")

    opts.add_argument("--window-size=1400,950")
    opts.add_argument("--lang=en-GB")
    opts.add_argument("--disable-gpu")
    opts.add_argument("--no-sandbox")

    opts.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )

    opts.add_experimental_option("excludeSwitches", ["enable-automation"])
    opts.add_experimental_option("useAutomationExtension", False)
    opts.add_argument("--disable-blink-features=AutomationControlled")

    driver = webdriver.Chrome(options=opts)
    try:
        driver.execute_cdp_cmd(
            "Page.addScriptToEvaluateOnNewDocument",
            {"source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined});"},
        )
    except Exception:
        pass

    return driver


# -------------------------
# CAPTCHA
# -------------------------
def is_funda_captcha_present(driver) -> bool:
    try:
        if driver.find_elements(By.CSS_SELECTOR, "form#fundaCaptchaForm"):
            return True
        if driver.find_elements(By.CSS_SELECTOR, "form[action*='/_akam_recaptcha_validate']"):
            return True
        html = (driver.page_source or "").lower()
        if "je bent bijna op de pagina die je zoekt" in html:
            return True
        if driver.find_elements(By.CSS_SELECTOR, "div.error-page") and "/_akam_recaptcha_validate" in html:
            return True
        return False
    except Exception:
        return False


def wait_captcha_if_present(driver, wanted_url: str | None = None, timeout_s: int = 300) -> None:
    if not is_funda_captcha_present(driver):
        return

    print("\n⚠️ Funda CAPTCHA gedetecteerd. Los 'm handmatig op in de browser...")
    start = time.time()
    while True:
        time.sleep(1.0)
        if not is_funda_captcha_present(driver):
            break
        if time.time() - start > timeout_s:
            raise TimeoutException(f"CAPTCHA niet opgelost binnen {timeout_s} seconden.")

    _try_accept_cookies(driver)
    time.sleep(0.7)
    if wanted_url:
        driver.get(wanted_url)
    else:
        driver.refresh()
    time.sleep(1.8)
    _try_accept_cookies(driver)
    time.sleep(0.5)


# -------------------------
# URL helpers
# -------------------------
def build_page_url(base_url: str, page_num: int) -> str:
    u = urlparse(base_url)
    q = parse_qs(u.query, keep_blank_values=True)
    q["page"] = [str(page_num)]
    new_query = urlencode(q, doseq=True)
    return urlunparse((u.scheme, u.netloc, u.path, u.params, new_query, u.fragment))


def _detail_base_url(detail_url: str) -> str:
    m = re.search(r"^(https?://www\.funda\.nl/en/detail/.+?/\d{6,})(?:/|$)", detail_url)
    if m:
        return m.group(1)
    return detail_url.rstrip("/")


def normalize_url(u: str, base: str = "https://www.funda.nl") -> str:
    if not u:
        return u
    u = u.strip()
    if u.startswith("//"):
        return "https:" + u
    if u.startswith("/"):
        return urljoin(base, u)
    return u


# -------------------------
# Search scrape
# -------------------------
def scrape_funda_search_links(
    search_url: str,
    max_pages: int = 10,
    max_scrolls_per_page: int = 8,
    pause_s: float = 1.0,
    max_links: int = 200,
    headless: bool = True,
) -> pd.DataFrame:
    driver = _make_driver(headless)
    try:
        links = set()
        no_new_pages_in_row = 0

        def grab_links() -> None:
            for a in driver.find_elements(By.CSS_SELECTOR, "a[href]"):
                href = a.get_attribute("href")
                if not href:
                    continue
                if re.search(r"https?://www\.funda\.nl/en/detail/[^/]+/.+/\d{6,}/?$", href):
                    links.add(href)

        def scroll_and_collect() -> None:
            last = -1
            for _ in range(max_scrolls_per_page):
                driver.execute_script("window.scrollBy(0, 3200);")
                time.sleep(pause_s)
                grab_links()
                if len(links) == last:
                    break
                last = len(links)

        for page_num in range(1, max_pages + 1):
            page_url = build_page_url(search_url, page_num)
            print(f"\n=== Open page={page_num} === {page_url}")

            before = len(links)

            driver.get(page_url)
            time.sleep(1.2)
            wait_captcha_if_present(driver, wanted_url=page_url, timeout_s=300)
            _try_accept_cookies(driver)

            driver.execute_script("window.scrollTo(0, 0);")
            time.sleep(0.4)

            grab_links()
            scroll_and_collect()

            after = len(links)
            print(f"  links totaal: {after} (nieuw op deze pagina: {after - before})")

            if after >= max_links:
                break

            if after == before:
                no_new_pages_in_row += 1
                if no_new_pages_in_row >= 2:
                    print("  (2 pagina's op rij geen nieuwe links -> stop)")
                    break
            else:
                no_new_pages_in_row = 0

            _sleep(0.7, 1.3)

        urls = sorted(links)[:max_links]
        df = pd.DataFrame({"detail_url": pd.Series(urls, dtype="string")})
        df["listing_id"] = df["detail_url"].str.extract(r"/(\d{6,})/?$")[0]
        return df
    finally:
        driver.quit()


# -------------------------
# Requests session (met selenium cookies)
# -------------------------
def _requests_session_from_driver(driver) -> requests.Session:
    sess = requests.Session()
    sess.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                      "(KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
        "Accept-Language": "en-GB,en;q=0.9,nl;q=0.8",
        "Accept": "image/avif,image/webp,image/apng,image/*,*/*;q=0.8",
        "Connection": "keep-alive",
    })
    try:
        for c in driver.get_cookies():
            sess.cookies.set(c.get("name"), c.get("value"))
    except Exception:
        pass
    return sess


# -------------------------
# Photo download (main image approach)
# -------------------------
def _parse_total_from_visible_text(driver) -> int | None:
    try:
        body_text = driver.find_element(By.TAG_NAME, "body").text.lower()
    except Exception:
        return None

    m = re.search(r"\b\d+\s*/\s*(\d{1,3})\b", body_text)
    if m:
        n = int(m.group(1))
        if 1 <= n <= 500:
            return n

    m = re.search(r"\b\d+\s+van\s+(\d{1,3})\b", body_text)
    if m:
        n = int(m.group(1))
        if 1 <= n <= 500:
            return n

    return None


def _pick_main_image_url(driver) -> str | None:
    imgs = driver.find_elements(By.CSS_SELECTOR, "img")
    best_area = -1
    best_url = None

    for img in imgs:
        try:
            if not img.is_displayed():
                continue
            rect = img.rect or {}
            w = float(rect.get("width") or 0)
            h = float(rect.get("height") or 0)
            area = w * h
            if area <= 10_000:
                continue

            cur = img.get_attribute("currentSrc") or ""
            src = img.get_attribute("src") or ""
            u = (cur.strip() or src.strip())
            u = normalize_url(u)
            if not u:
                continue

            p = urlparse(u).path.lower()
            if not (p.endswith(".jpg") or p.endswith(".jpeg") or p.endswith(".png") or p.endswith(".webp")):
                continue

            if area > best_area:
                best_area = area
                best_url = u
        except Exception:
            continue

    return best_url


def _download_one(session: requests.Session, url: str, out_file: Path, referer: str, timeout_s: int = 60) -> tuple[bool, int]:
    try:
        headers = {"Referer": referer}
        with session.get(url, timeout=timeout_s, headers=headers, stream=True) as r:
            status = r.status_code
            if status != 200:
                return False, status
            with open(out_file, "wb") as f:
                for chunk in r.iter_content(chunk_size=1024 * 128):
                    if chunk:
                        f.write(chunk)
        return True, 200
    except Exception:
        return False, -1


def download_listing_photos_streaming(
    driver,
    session: requests.Session,
    detail_url: str,
    listing_id: str,
    out_root: Path,
    max_photos_fallback: int = 120,
    miss_limit: int = 6,
    first_only: bool = False,   # <-- NIEUW
) -> tuple[int, int]:
    base = _detail_base_url(detail_url)
    listing_dir = out_root / str(listing_id)
    _safe_mkdir(listing_dir)

    total: int | None = None
    found = 0
    saved = 0
    misses = 0

    max_i = 1 if first_only else max_photos_fallback

    for i in range(1, max_i + 1):
        media_url = f"{base}/media/foto/{i}"
        driver.get(media_url)
        time.sleep(0.9)
        wait_captcha_if_present(driver, wanted_url=media_url, timeout_s=300)
        _try_accept_cookies(driver)
        time.sleep(0.3)

        if i == 1 and not first_only:
            total = _parse_total_from_visible_text(driver)
            if total:
                print(f"  totaal foto's (uit teller): {total}")
            else:
                print(f"  totaal foto's niet gevonden, fallback max={max_photos_fallback} + miss_limit={miss_limit}")

        if (not first_only) and total is not None and i > total:
            break

        img_url = _pick_main_image_url(driver)
        if not img_url:
            misses += 1
            print(f"  foto/{i}: geen main image gevonden (miss {misses}/{miss_limit})")
            if (not first_only) and total is None and misses >= miss_limit:
                break
            continue

        misses = 0
        found += 1
        print(f"  foto/{i}: gevonden")

        ext = os.path.splitext(urlparse(img_url).path)[1].lower()
        if ext not in {".jpg", ".jpeg", ".png", ".webp"}:
            ext = ".jpg"
        out_file = listing_dir / f"{listing_id}_{i:02d}{ext}"

        ok, status = _download_one(session, img_url, out_file, referer=media_url, timeout_s=60)
        if ok:
            saved += 1
            print(f"    ✓ opgeslagen: {out_file.name}")
        else:
            print(f"    ✗ HTTP {status} bij download -> niet opgeslagen")
            print(f"      url: {img_url}")

        _sleep(0.15, 0.45)

    return found, saved


# -------------------------
# Features scrape -> JSON
# -------------------------
def _clean_text(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "").strip())


def scrape_features(driver, detail_url: str) -> dict:
    driver.get(detail_url)
    time.sleep(1.2)
    wait_captcha_if_present(driver, wanted_url=detail_url, timeout_s=300)
    _try_accept_cookies(driver)
    time.sleep(0.6)

    sections: dict[str, dict[str, str]] = {}

    feature_root_candidates = driver.find_elements(
        By.XPATH,
        "//*[self::h2 or self::h1][contains(translate(.,'FEATURES','features'),'features')]"
    )
    root = feature_root_candidates[0] if feature_root_candidates else None

    container = None
    if root:
        try:
            container = root.find_element(By.XPATH, "./ancestor::*[self::section or self::div][1]")
        except Exception:
            container = None

    search_scope = container if container else driver

    headings = search_scope.find_elements(By.XPATH, ".//*[self::h3 or self::h4]")
    for h in headings:
        title = _clean_text(h.text)
        if not title:
            continue

        try:
            block = h.find_element(By.XPATH, "following::*[self::table or self::dl][1]")
        except Exception:
            block = None
        if not block:
            continue

        kv: dict[str, str] = {}
        tag = (block.tag_name or "").lower()

        if tag == "table":
            rows = block.find_elements(By.XPATH, ".//tr")
            for r in rows:
                cells = r.find_elements(By.XPATH, ".//th|.//td")
                if len(cells) >= 2:
                    k = _clean_text(cells[0].text)
                    v = _clean_text(cells[1].text)
                    if k:
                        kv[k] = v

        elif tag == "dl":
            dts = block.find_elements(By.XPATH, ".//dt")
            for dt in dts:
                try:
                    dd = dt.find_element(By.XPATH, "following-sibling::dd[1]")
                except Exception:
                    dd = None
                k = _clean_text(dt.text)
                v = _clean_text(dd.text if dd else "")
                if k:
                    kv[k] = v

        if kv:
            sections[title] = kv

    return {"url": detail_url, "sections": sections}


def save_features_json(out_dir: Path, listing_id: str, data: dict) -> Path:
    listing_dir = out_dir / str(listing_id)
    _safe_mkdir(listing_dir)
    fn = listing_dir / "features.json"
    fn.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    return fn


# -------------------------
# Skip logic
# -------------------------
def should_skip_listing(out_dir: Path, listing_id: str, skip_if_features_json_exists: bool = True) -> bool:
    listing_dir = out_dir / str(listing_id)
    features_json = listing_dir / "features.json"
    if skip_if_features_json_exists and features_json.exists():
        try:
            return features_json.stat().st_size > 10
        except Exception:
            return True
    return False


# -------------------------
# Main
# -------------------------
def main(
    search_url: str = SEARCH_URL,
    out_root: str = "funda_photos",
    max_listings: int = 200,
    max_pages: int = 15,
    headless: bool = False,
    also_scrape_features: bool = True,
    skip_if_features_json_exists: bool = True,
    first_only: bool = False,  # <-- NIEUW
):
    out_dir = Path(out_root)
    _safe_mkdir(out_dir)

    df = scrape_funda_search_links(
        search_url,
        max_pages=max_pages,
        max_scrolls_per_page=10,
        pause_s=1.0,
        max_links=max_listings,
        headless=headless,
    )
    print(f"\ngevonden listings totaal: {len(df)}")

    links_csv = out_dir / "links.csv"
    df.to_csv(links_csv, index=False)
    print(f"links opgeslagen: {links_csv}")

    driver = _make_driver(headless)
    rows = []
    try:
        driver.get("https://www.funda.nl/")
        time.sleep(1.5)
        wait_captcha_if_present(driver, wanted_url="https://www.funda.nl/", timeout_s=300)
        _try_accept_cookies(driver)

        for idx, row in df.iterrows():
            listing_id = row["listing_id"]
            detail_url = row["detail_url"]
            if not isinstance(listing_id, str) or not isinstance(detail_url, str):
                continue

            if should_skip_listing(out_dir, listing_id, skip_if_features_json_exists=skip_if_features_json_exists):
                print(f"\n[{idx+1}/{len(df)}] listing_id={listing_id} -> SKIP (features.json bestaat al)")
                rows.append({
                    "listing_id": listing_id,
                    "detail_url": detail_url,
                    "photo_count_found": 0,
                    "photo_count_saved": 0,
                    "features_json": str((out_dir / listing_id / "features.json")),
                    "skipped": True,
                })
                continue

            print(f"\n[{idx+1}/{len(df)}] listing_id={listing_id}")

            session = _requests_session_from_driver(driver)

            found, saved = download_listing_photos_streaming(
                driver=driver,
                session=session,
                detail_url=detail_url,
                listing_id=listing_id,
                out_root=out_dir,
                max_photos_fallback=120,
                miss_limit=6,
                first_only=first_only,
            )

            features_path = ""
            if also_scrape_features:
                feats = scrape_features(driver, detail_url)
                fp = save_features_json(out_dir, listing_id, feats)
                features_path = str(fp)
                print(f"  ✓ features opgeslagen: {fp}")

            print(f"  -> found={found}, saved={saved}")

            rows.append({
                "listing_id": listing_id,
                "detail_url": detail_url,
                "photo_count_found": found,
                "photo_count_saved": saved,
                "features_json": features_path,
                "skipped": False,
            })

            _sleep(1.2, 2.2)

    finally:
        driver.quit()

    log_df = pd.DataFrame(rows)
    log_csv = out_dir / "download_log.csv"
    log_df.to_csv(log_csv, index=False)
    print(f"\nKlaar. Log: {log_csv}")


if __name__ == "__main__":
    main(
        search_url=SEARCH_URL,
        out_root="funda_photos",
        max_listings=50000,
        max_pages=666,
        headless=False,
        also_scrape_features=True,
        skip_if_features_json_exists=True,
        first_only=True,   # <-- zet True voor alleen eerste foto
    )


=== Open page=1 === https://www.funda.nl/en/zoeken/koop?object_type=%5B%22house%22%5D&availability=%5B%22available%22%5D&page=1
  links totaal: 18 (nieuw op deze pagina: 18)

=== Open page=2 === https://www.funda.nl/en/zoeken/koop?object_type=%5B%22house%22%5D&availability=%5B%22available%22%5D&page=2
  links totaal: 36 (nieuw op deze pagina: 18)

=== Open page=3 === https://www.funda.nl/en/zoeken/koop?object_type=%5B%22house%22%5D&availability=%5B%22available%22%5D&page=3
  links totaal: 54 (nieuw op deze pagina: 18)

=== Open page=4 === https://www.funda.nl/en/zoeken/koop?object_type=%5B%22house%22%5D&availability=%5B%22available%22%5D&page=4
  links totaal: 72 (nieuw op deze pagina: 18)

=== Open page=5 === https://www.funda.nl/en/zoeken/koop?object_type=%5B%22house%22%5D&availability=%5B%22available%22%5D&page=5
  links totaal: 90 (nieuw op deze pagina: 18)

=== Open page=6 === https://www.funda.nl/en/zoeken/koop?object_type=%5B%22house%22%5D&availability=%5B%22available%22%5D&pag

KeyboardInterrupt: 